In [1]:
import os, zipfile, warnings
import numpy as np
import pandas as pd
import joblib
from sklearn.model_selection import train_test_split
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

In [2]:
with np.load("mnist.npz") as f:
    X_train_full, y_train_full = f["x_train"], f["y_train"]
    X_test, y_test = f["x_test"], f["y_test"]

# reshape если данные плоские
if X_train_full.ndim == 2:
    X_train_full = X_train_full.reshape((-1, 28, 28))
if X_test.ndim == 2:
    X_test = X_test.reshape((-1, 28, 28))

# делаем векторы
X_train_full = X_train_full.reshape(len(X_train_full), -1)
X_test = X_test.reshape(len(X_test), -1)

X_train_full = X_train_full.astype(np.float32) / 255.0
X_test = X_test.astype(np.float32) / 255.0
y_train_full = y_train_full.astype(int)
y_test = y_test.astype(int)

print("Train:", X_train_full.shape, "Test:", X_test.shape)

Train: (60000, 784) Test: (10000, 784)


In [3]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_full)
X_test_scaled = scaler.transform(X_test)

pca = PCA(n_components=0.95, svd_solver="full", random_state=42)
X_train_pca = pca.fit_transform(X_train_scaled)
X_test_pca = pca.transform(X_test_scaled)

print("PCA размерность:", X_train_pca.shape[1])

PCA размерность: 331


In [4]:
rf = RandomForestClassifier(n_estimators=200, max_depth=None, n_jobs=-1, random_state=42)
rf.fit(X_train_pca, y_train_full)

y_pred = rf.predict(X_test_pca)
acc = accuracy_score(y_test, y_pred)
print("Test accuracy:", acc)
print(classification_report(y_test, y_pred))

Test accuracy: 0.9406
              precision    recall  f1-score   support

           0       0.96      0.98      0.97       980
           1       0.99      0.98      0.98      1135
           2       0.93      0.93      0.93      1032
           3       0.89      0.93      0.91      1010
           4       0.94      0.95      0.95       982
           5       0.94      0.91      0.93       892
           6       0.95      0.96      0.96       958
           7       0.93      0.93      0.93      1028
           8       0.94      0.92      0.93       974
           9       0.93      0.90      0.91      1009

    accuracy                           0.94     10000
   macro avg       0.94      0.94      0.94     10000
weighted avg       0.94      0.94      0.94     10000



In [5]:
submission = pd.DataFrame({"index": np.arange(len(y_pred)), "label": y_pred})
submission.to_csv("submission.csv", index=False)